In [1]:
%pip install natsort
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import re
from natsort import natsorted

# =============================================================================
# 1. 参数区
# =============================================================================
folder_path = Path(r"D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007")
SOH_FILENAME_PATTERN = re.compile(r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$", flags=re.IGNORECASE)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(filename.strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))

# 实验设置的SOC顺序
SOC_ORDER = ["90%", "50%", "10%"]

# cycle内脉冲按照ID划分
REMOVE_PULSE_BEFORE_MIN = 60

# 实测 active 会比 3h 稍大，但不会超过 CYCLE_ACTIVE_LIMIT_HOUR.
# 因此用 CYCLE_ACTIVE_LIMIT_HOUR 作为 time_diff 判断是否进入下一 cycle 的边界。
CYCLE_ACTIVE_LIMIT_HOUR = 4.0

# 设置电流标准差
STD_LIMIT_1P5A = 0.1
STD_LIMIT_3A = 0.1
NOMINAL_CURRENT_LEVELS_A = np.array([1.5, 3.0])
CC_CURRENT_DEVIATION_RATIO = 0.03  # 10s/20s前相对早期稳定电流的最大允许偏差
CC_STATUS_VALID = "Valid"
CC_STATUS_INSUFFICIENT = "Insufficient duration"
CC_STATUS_DEVIATION = "Current deviation > 3%"
CC_STATUS_NOT_EVALUATED = "Not evaluated"

# R0 / R1 / R2计算设置
R0_TARGET_AFTER_PAUSE_SEC = 0.5   # pause段最后一个测量点之后外推的R0时间
R1_ANCHOR_SEC = 10.0              # 相对于首个非零电流点
R2_ANCHOR_SEC = 20.0              # 相对于首个非零电流点
R0_FIT_POINT_START = 2            # 默认使用有效pulse第2-6点
R0_FIT_POINT_END = 6              # 拟合终点固定为有效pulse第6点
FIRST_TWO_VOLTAGE_EQUAL_ATOL = 1e-9  # 前两点电压视为相同的绝对容差(V)
ZERO_CURRENT_LIMIT = 1e-6
VOLTAGE_JUMP_LIMIT = 1e-3
MAX_PAUSE_TO_PULSE_GAP_SEC = 5.0  # pause末点到pulse首点严格大于5s则剔除
R0_EARLY_WINDOW_FLAG = "后段电流不稳定，R0仅使用早期稳定窗口"

# R0置信度规则：
# “干净标签”的有效R0可按权重1.0使用。
TRUSTED_R0_QUALITY_FLAGS = {
    "正常",
    R0_EARLY_WINDOW_FLAG,
    "首点0且电压跳变"
}

R0_CONFIDENCE_FULL = "完全可信（权重1.0）"
R0_CONFIDENCE_REVIEW = "需复核（权重0.5）"
R0_CONFIDENCE_INVALID = "不可用（权重0.0）"

TIME_DIFF_OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current",
    "Voltage",
    "Zustand",
    "ID",
    "Zustand/Current"
]

# R0、R1、R2及质量检验列
PULSE_OUTPUT_COLUMNS = TIME_DIFF_OUTPUT_COLUMNS + [
    "Nominal_Current_A",
    "R0",
    "R1",
    "R2",
    "Temperature",
    "T_start",
    "T_end",
    "T_mean",
    "Delta_T",
    "CC_Valid_10s",
    "CC_Valid_20s",
    "CC_10s_Status",
    "CC_20s_Status",
    "CC_Max_Deviation_10s_pct",
    "CC_Max_Deviation_20s_pct",
    "R0_Target_Time",
    "Pause_to_Pulse_Time_Diff_s",
    "R0_Quality",
    "R0_Confidence"
]


Note: you may need to restart the kernel to use updated packages.


d:\python\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\python\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
# =============================================================================
# 2. 读取 parquet
# =============================================================================

parquet_files = natsorted(list(folder_path.rglob("*.parquet")))

if len(parquet_files) == 0:
    raise FileNotFoundError(f"没有在文件夹中找到 parquet 文件: {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name
    temp["SOH"] = extract_soh_from_filename(file.name)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [3]:
# =============================================================================
# 3. time diff 主函数
# =============================================================================

def build_time_diff_sequence(df):
    df_td = df.copy()

    # -----------------------------
    # 1. 基础时间、电流、电压处理
    # -----------------------------
    df_td["Time"] = pd.to_datetime(df_td["Time"], utc=True, errors="coerce")
    df_td["Current"] = pd.to_numeric(df_td["Current"], errors="coerce")
    df_td["Voltage"] = pd.to_numeric(df_td["Voltage"], errors="coerce")
    df_td["Temperature"] = pd.to_numeric(df_td["Temperature"], errors="coerce")

    df_td = df_td.dropna(subset=["Time", "Current"]).copy()
    df_td = df_td.sort_values(["File", "Time"]).reset_index(drop=True)

    # -----------------------------
    # 2. 按 time_diff 划分 cycle / SOC
    # -----------------------------
    # 直接比较同一个 File 内相邻时间点的时间差：
    #   time_diff <= CYCLE_ACTIVE_LIMIT_HOUR：仍属于当前 cycle
    #   time_diff >  CYCLE_ACTIVE_LIMIT_HOUR：说明中间经过 pause，进入下一个 cycle
    df_td["time_diff_hour"] = (
        df_td.groupby("File")["Time"].diff() / pd.Timedelta(hours=1)
    )

    df_td["is_new_cycle"] = (
        df_td["time_diff_hour"].isna()
        | (df_td["time_diff_hour"] > CYCLE_ACTIVE_LIMIT_HOUR)
    )

    df_td["cycle_id"] = (
        df_td.groupby("File")["is_new_cycle"]
        .cumsum()
        .astype(int)
    )

    df_td["SOC"] = df_td["cycle_id"].map(
        lambda cycle_id: SOC_ORDER[(cycle_id - 1) % len(SOC_ORDER)]
    )

    cycle_start_time = df_td.groupby(["File", "cycle_id"])["Time"].transform("min")

    df_td["time_from_cycle_start_min"] = (
        df_td["Time"] - cycle_start_time
    ) / pd.Timedelta(minutes=1)

    # -----------------------------
    # 3. 统一 Zustand
    # -----------------------------
    df_td["Zustand"] = df_td["Zustand"].astype(str)

    df_td.loc[
        df_td["Zustand"].str.startswith("DCH", na=False),
        "Zustand"
    ] = "DCH"

    df_td.loc[
        df_td["Zustand"].str.startswith("CHA", na=False),
        "Zustand"
    ] = "CHA"

    # -----------------------------
    # 4. 生成 pulse_segment_id
    # -----------------------------
    df_td["pulse_segment_id"] = (
        df_td["File"].ne(df_td["File"].shift())
        | df_td["Zustand"].ne(df_td["Zustand"].shift())
    ).cumsum()

    # -----------------------------
    # 5. 生成 Zustand/Current，并保留完整 time_diff_sequence
    # -----------------------------
    df_td["Zustand/Current"] = (
        df_td["Zustand"]
        + "/"
        + df_td["Current"].astype(float).round(1).astype(str)
    )

    # 这个表保留所有状态点，后面用于筛选 PAUO
    time_diff_sequence = df_td[TIME_DIFF_OUTPUT_COLUMNS].copy()

    # 只保留 CHA / DCH 作为 pulse_sequence
    pulse_mask = df_td["Zustand"].str.startswith(("CHA", "DCH"), na=False)
    pulse_sequence = df_td[pulse_mask].copy()
    pulse_sequence = pulse_sequence.sort_values(
        ["File", "pulse_segment_id", "Time"]
    )

    # -----------------------------
    # 6. 剔除电流不稳定的 pulse_segment_id
    # -----------------------------
    def get_effective_start_pos(group):
    # 返回pulse段内第一个非零有效电流点的位置。
        current_abs = group["Current"].abs().to_numpy()
        non_zero_pos = np.flatnonzero(current_abs > ZERO_CURRENT_LIMIT)

        if len(non_zero_pos) == 0:
            return None

        return int(non_zero_pos[0])

    def is_bad_current_segment(group):
        group = group.sort_values("Time")

        effective_start_pos = get_effective_start_pos(group)
        if effective_start_pos is None:
            return True

        # 如果首个测量点 Current == 0，则从第一个非零有效点开始判断稳定性
        current_values = group["Current"].iloc[effective_start_pos:]

        current_abs_level = round(current_values.abs().iloc[0], 1)
        current_std = current_values.std()

        if current_abs_level == 1.5:
            return current_std > STD_LIMIT_1P5A

        if current_abs_level == 3.0:
            return current_std > STD_LIMIT_3A

        return True

    # 不再因为pulse后段电流下降而提前删除整段数据。
    # R0是否可计算，改为在下面仅依据实际拟合窗口内的电流稳定性判断。

    # -----------------------------
    # 7. 计算R0，并且每个 pulse_segment_id 只取一个代表点
    # -----------------------------
    def add_quality(base_quality, new_quality):
        if base_quality == "正常":
            return new_quality
        return base_quality + "；" + new_quality

    def calculate_r0_for_segment(group, effective_start_pos):
        group = group.sort_values("Time")

        r0_result = {
            "Nominal_Current_A": np.nan,
            "R0": np.nan,
            "R1": np.nan,
            "R2": np.nan,
            "Temperature": np.nan,
            "T_start": np.nan,
            "T_end": np.nan,
            "T_mean": np.nan,
            "Delta_T": np.nan,
            "CC_Valid_10s": pd.NA,
            "CC_Valid_20s": pd.NA,
            "CC_10s_Status": CC_STATUS_NOT_EVALUATED,
            "CC_20s_Status": CC_STATUS_NOT_EVALUATED,
            "CC_Max_Deviation_10s_pct": np.nan,
            "CC_Max_Deviation_20s_pct": np.nan,
            "R0_Target_Time": pd.NaT,
            "Pause_to_Pulse_Time_Diff_s": np.nan,
            "R0_Quality": "正常"
        }

        file_name = group["File"].iloc[0]
        pulse_start_time = group["Time"].iloc[0]
        first_current = group["Current"].iloc[0]
        segment_current_abs = group["Current"].abs().dropna()
        temperature_values = group["Temperature"].dropna()

        if not temperature_values.empty:
            t_start = float(temperature_values.iloc[0])
            t_end = float(temperature_values.iloc[-1])
            r0_result["Temperature"] = float(temperature_values.median())
            r0_result["T_start"] = t_start
            r0_result["T_end"] = t_end
            r0_result["T_mean"] = (t_start + t_end) / 2
            r0_result["Delta_T"] = t_end - t_start

        if not segment_current_abs.empty:
            measured_peak_current = segment_current_abs.max()
            nearest_nominal_pos = np.argmin(
                np.abs(NOMINAL_CURRENT_LEVELS_A - measured_peak_current)
            )
            r0_result["Nominal_Current_A"] = float(
                NOMINAL_CURRENT_LEVELS_A[nearest_nominal_pos]
            )

        previous_pause = df_td[
            (df_td["File"] == file_name)
            & (df_td["Time"] < pulse_start_time)
            & (df_td["Zustand"].str.startswith("PAU", na=False))
        ].sort_values("Time").tail(1)

        if previous_pause.empty:
            r0_result["R0_Quality"] = "无法计算R0：无前置pause点"
            return r0_result

        pause_time = previous_pause["Time"].iloc[0]
        pause_voltage = previous_pause["Voltage"].iloc[0]

        pause_to_pulse_gap_s = (pulse_start_time - pause_time).total_seconds()
        r0_result["Pause_to_Pulse_Time_Diff_s"] = pause_to_pulse_gap_s

        target_time = pause_time + pd.Timedelta(seconds=R0_TARGET_AFTER_PAUSE_SEC)
        r0_result["R0_Target_Time"] = target_time

        # 大于5s：标记为无效，不再进行长距离反向外推
        if pause_to_pulse_gap_s > MAX_PAUSE_TO_PULSE_GAP_SEC:
            r0_result["R0_Quality"] = "pause结束点到pulse首点时间差>5s"
            return r0_result

        # pulse段第一个点为0时：无论voltage是否跳变，R0计算都从第一个非零有效点开始；
        # 若voltage已经跳变，则额外给质量标记。
        if abs(first_current) <= ZERO_CURRENT_LIMIT:
            first_voltage = group["Voltage"].iloc[0]

            if (
                pd.notna(first_voltage)
                and pd.notna(pause_voltage)
                and abs(first_voltage - pause_voltage) > VOLTAGE_JUMP_LIMIT
            ):
                r0_result["R0_Quality"] = "首点0且电压跳变"

        # 默认使用有效pulse第2-6点进行线性拟合。
        # 若有效pulse第1点与第2点的电压相同（可能是重复采样），
        # 则跳过前两点，改用第3-6点进行线性外推。
        fit_point_start = R0_FIT_POINT_START

        if effective_start_pos + 1 < len(group):
            first_pulse_voltage = group["Voltage"].iloc[effective_start_pos]
            second_pulse_voltage = group["Voltage"].iloc[effective_start_pos + 1]

            first_two_voltage_equal = (
                pd.notna(first_pulse_voltage)
                and pd.notna(second_pulse_voltage)
                and np.isclose(
                    float(first_pulse_voltage),
                    float(second_pulse_voltage),
                    rtol=0.0,
                    atol=FIRST_TWO_VOLTAGE_EQUAL_ATOL
                )
            )

            if first_two_voltage_equal:
                fit_point_start = 3

        fit_start_pos = effective_start_pos + fit_point_start - 1
        fit_end_pos = effective_start_pos + R0_FIT_POINT_END

        fit_points = group.iloc[fit_start_pos:fit_end_pos].dropna(
            subset=["Time", "Voltage", "Current"]
        )

        if len(fit_points) < 2:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：拟合点不足"
            )
            return r0_result

        # R0只要求实际参与2-6点（或3-6点）拟合的早期窗口电流稳定。
        fit_current_values = fit_points["Current"]
        fit_current_abs_level = round(fit_current_values.abs().iloc[0], 1)
        fit_current_std = fit_current_values.std()

        if fit_current_abs_level == 1.5:
            fit_current_unstable = fit_current_std > STD_LIMIT_1P5A
        elif fit_current_abs_level == 3.0:
            fit_current_unstable = fit_current_std > STD_LIMIT_3A
        else:
            fit_current_unstable = True

        if fit_current_unstable:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：早期拟合窗口电流不稳定"
            )
            return r0_result

        # 若整段电流不稳定、但早期拟合窗口稳定，仍计算R0并添加专用flag。
        if is_bad_current_segment(group):
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                R0_EARLY_WINDOW_FLAG
            )

        effective_current = group["Current"].iloc[effective_start_pos]

        if abs(effective_current) <= ZERO_CURRENT_LIMIT:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：无非零有效电流"
            )
            return r0_result

        if pd.isna(pause_voltage):
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：pause电压缺失"
            )
            return r0_result
        
        # 判断采样点距离pulse起点的距离
        x_sec = (fit_points["Time"] - target_time) / pd.Timedelta(seconds=1)

        y_voltage = fit_points["Voltage"].astype(float)

        slope, intercept = np.polyfit(x_sec.to_numpy(), y_voltage.to_numpy(), 1)
        extrapolated_voltage = intercept

        r0_result["R0"] = abs(
            (extrapolated_voltage - pause_voltage) / effective_current
        )

        # 以首个非零电流点为t=0。CC检查从已通过R0标准差检验的
        # 早期拟合窗口开始，避免把电流阶跃建立过程误判成进入CV。
        anchor_data = group.iloc[effective_start_pos:].dropna(
            subset=["Time", "Voltage", "Current"]
        )
        relative_time_s = (
            anchor_data["Time"] - group["Time"].iloc[effective_start_pos]
        ) / pd.Timedelta(seconds=1)

        early_current_ref = fit_current_values.median()
        fit_start_relative_s = (
            fit_points["Time"].min()
            - group["Time"].iloc[effective_start_pos]
        ) / pd.Timedelta(seconds=1)

        def evaluate_cc_until(anchor_sec):
            if len(anchor_data) < 2 or relative_time_s.max() < anchor_sec:
                return CC_STATUS_INSUFFICIENT, np.nan

            if (
                fit_start_relative_s >= anchor_sec
                or pd.isna(early_current_ref)
                or abs(early_current_ref) <= ZERO_CURRENT_LIMIT
            ):
                return CC_STATUS_NOT_EVALUATED, np.nan

            window_mask = (
                (relative_time_s >= fit_start_relative_s)
                & (relative_time_s <= anchor_sec)
            )
            window_current = anchor_data.loc[window_mask, "Current"].to_numpy()
            anchor_current = np.interp(
                anchor_sec,
                relative_time_s.to_numpy(),
                anchor_data["Current"].to_numpy()
            )
            current_values = np.append(window_current, anchor_current)
            if len(current_values) < 2:
                return CC_STATUS_NOT_EVALUATED, np.nan

            max_deviation_pct = (
                np.max(np.abs(current_values - early_current_ref))
                / abs(early_current_ref)
                * 100
            )

            if max_deviation_pct <= CC_CURRENT_DEVIATION_RATIO * 100:
                return CC_STATUS_VALID, max_deviation_pct

            return CC_STATUS_DEVIATION, max_deviation_pct

        cc_10s_status, cc_max_deviation_10s_pct = evaluate_cc_until(
            R1_ANCHOR_SEC
        )
        cc_20s_status, cc_max_deviation_20s_pct = evaluate_cc_until(
            R2_ANCHOR_SEC
        )
        cc_valid_10s = cc_10s_status == CC_STATUS_VALID
        cc_valid_20s = cc_20s_status == CC_STATUS_VALID
        r0_result["CC_Valid_10s"] = cc_valid_10s
        r0_result["CC_Valid_20s"] = cc_valid_20s
        r0_result["CC_10s_Status"] = cc_10s_status
        r0_result["CC_20s_Status"] = cc_20s_status
        r0_result["CC_Max_Deviation_10s_pct"] = cc_max_deviation_10s_pct
        r0_result["CC_Max_Deviation_20s_pct"] = cc_max_deviation_20s_pct

        if cc_valid_10s:
            voltage_10s = np.interp(
                R1_ANCHOR_SEC,
                relative_time_s.to_numpy(),
                anchor_data["Voltage"].to_numpy()
            )
            r10 = abs((voltage_10s - pause_voltage) / effective_current)
            r0_result["R1"] = r10 - r0_result["R0"]

        if cc_valid_20s:
            voltage_10s, voltage_20s = np.interp(
                [R1_ANCHOR_SEC, R2_ANCHOR_SEC],
                relative_time_s.to_numpy(),
                anchor_data["Voltage"].to_numpy()
            )
            r10 = abs((voltage_10s - pause_voltage) / effective_current)
            r20 = abs((voltage_20s - pause_voltage) / effective_current)
            r0_result["R2"] = r20 - r10

        return r0_result

    selected_indices = []
    r0_results = {}

    for _, group in pulse_sequence.groupby(["File", "pulse_segment_id"], sort=False):
        group = group.sort_values("Time")

        effective_start_pos = get_effective_start_pos(group)
        if effective_start_pos is None:
            continue

        # 在计算R0前剔除极端工况：10% SOC下的3.0A DCH，
        # 以及90% SOC下的3.0A CHA。
        effective_current_abs = round(
            abs(group["Current"].iloc[effective_start_pos]),
            1
        )
        soc = group["SOC"].iloc[0]
        zustand = group["Zustand"].iloc[0]

        if effective_current_abs == 3.0 and (
            (soc == "10%" and zustand == "DCH")
            or (soc == "90%" and zustand == "CHA")
        ):
            continue

        # 保留原逻辑：每段用“有效pulse起点后的第2个测量点”记录；
        # 如果点数不够，则退回到有效pulse起点本身。
        record_pos = effective_start_pos + 1
        if record_pos >= len(group):
            record_pos = effective_start_pos

        record_index = group.index[record_pos]
        selected_indices.append(record_index)

        r0_results[record_index] = calculate_r0_for_segment(
            group,
            effective_start_pos
        )

    pulse_sequence = pulse_sequence.loc[selected_indices].copy()

    r0_result_columns = [
        "Nominal_Current_A",
        "R0",
        "R1",
        "R2",
        "Temperature",
        "T_start",
        "T_end",
        "T_mean",
        "Delta_T",
        "CC_Valid_10s",
        "CC_Valid_20s",
        "CC_10s_Status",
        "CC_20s_Status",
        "CC_Max_Deviation_10s_pct",
        "CC_Max_Deviation_20s_pct",
        "R0_Target_Time",
        "Pause_to_Pulse_Time_Diff_s",
        "R0_Quality"
    ]

    for column in r0_result_columns:
        pulse_sequence[column] = pulse_sequence.index.map(
            lambda idx, col=column: r0_results[idx][col]
        )

    pulse_sequence = pulse_sequence.reset_index(drop=True)

    # -----------------------------
    # 8. 生成逐条R0置信度
    # -----------------------------
    def classify_r0_confidence(row):
        r0_value = row["R0"]
        quality_text = str(row["R0_Quality"]).strip()

        # 没有有效R0时，无论带有什么flag，都不能参与R0分析。
        if pd.isna(r0_value) or not np.isfinite(r0_value):
            return R0_CONFIDENCE_INVALID

        flags = {
            flag.strip()
            for flag in quality_text.split("；")
            if flag.strip()
        }

        # 只要一条记录所含flag全部属于已验证的干净标签，就给权重1.0。
        if flags and flags.issubset(TRUSTED_R0_QUALITY_FLAGS):
            return R0_CONFIDENCE_FULL

        # 将来若新增了仍可计算R0、但尚未验证的flag，先标记为需复核。
        return R0_CONFIDENCE_REVIEW

    pulse_sequence["R0_Confidence"] = pulse_sequence.apply(
        classify_r0_confidence,
        axis=1
    )

    # -----------------------------
    # 9. 只保留最终输出列
    # -----------------------------
    pulse_sequence = pulse_sequence[PULSE_OUTPUT_COLUMNS].copy()

    return pulse_sequence, time_diff_sequence

pulse_sequence, time_diff_sequence = build_time_diff_sequence(df)

# 中间结果预览；统一的R0_Quality统计放在最终绘图cell中，避免重复输出。
display(pulse_sequence)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,Nominal_Current_A,...,CC_Valid_10s,CC_Valid_20s,CC_10s_Status,CC_20s_Status,CC_Max_Deviation_10s_pct,CC_Max_Deviation_20s_pct,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 12:03:52.760000+00:00,-1.499470,4.036492,DCH,12_17,DCH/-1.5,1.5,...,<NA>,<NA>,Not evaluated,Not evaluated,NaN,NaN,NaT,NaN,无法计算R0：无前置pause点,不可用（权重0.0）
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5,1.5,...,True,True,Valid,Valid,0.035993,0.035993,2024-11-12 13:04:35.250000+00:00,0.76,正常,完全可信（权重1.0）
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0,3.0,...,True,True,Valid,Valid,0.047994,0.047994,2024-11-12 14:05:18.390000+00:00,0.81,正常,完全可信（权重1.0）
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 19:55:56.640000+00:00,-1.498840,3.725950,DCH,12_35,DCH/-1.5,1.5,...,<NA>,<NA>,Not evaluated,Not evaluated,NaN,NaN,2024-11-12 15:36:43.330000+00:00,15553.57,pause结束点到pulse首点时间差>5s,不可用（权重0.0）
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5,1.5,...,True,True,Valid,Valid,0.017994,0.017994,2024-11-12 20:56:39.410000+00:00,0.81,正常,完全可信（权重1.0）
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.530000+00:00,-2.999504,3.717166,DCH,9_44,DCH/-3.0,3.0,...,True,True,Valid,Valid,0.011993,0.011993,2024-10-23 22:29:48.980000+00:00,0.84,正常,完全可信（权重1.0）
233,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0,3.0,...,True,True,Valid,Valid,0.006000,0.006000,2024-10-23 23:30:52.300000+00:00,0.81,正常,完全可信（权重1.0）
234,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 04:22:48.810000+00:00,-1.497851,3.192480,DCH,9_54,DCH/-1.5,1.5,...,<NA>,<NA>,Not evaluated,Not evaluated,NaN,NaN,2024-10-24 00:01:13.750000+00:00,15695.30,pause结束点到pulse首点时间差>5s,不可用（权重0.0）
235,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5,1.5,...,True,True,Valid,Valid,0.065992,0.065992,2024-10-24 05:23:31.420000+00:00,0.88,正常,完全可信（权重1.0）


In [4]:
# R0 计算部分
# -----------------------------
# 9. 筛选脉冲/清除1.5A下的DCH脉冲
# -----------------------------
def filter_pulse(df):

    pulse_sequence_filter = df[
        ~((df["Zustand"] == "DCH") & (df["Nominal_Current_A"] == 1.5))
    ].copy()
    return pulse_sequence_filter

filtered_pulse = filter_pulse(pulse_sequence)
display(filtered_pulse)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,Nominal_Current_A,...,CC_Valid_10s,CC_Valid_20s,CC_10s_Status,CC_20s_Status,CC_Max_Deviation_10s_pct,CC_Max_Deviation_20s_pct,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5,1.5,...,True,True,Valid,Valid,0.035993,0.035993,2024-11-12 13:04:35.250000+00:00,0.76,正常,完全可信（权重1.0）
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0,3.0,...,True,True,Valid,Valid,0.047994,0.047994,2024-11-12 14:05:18.390000+00:00,0.81,正常,完全可信（权重1.0）
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5,1.5,...,True,True,Valid,Valid,0.017994,0.017994,2024-11-12 20:56:39.410000+00:00,0.81,正常,完全可信（权重1.0）
5,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.250000+00:00,-2.999144,3.719279,DCH,12_43,DCH/-3.0,3.0,...,True,True,Valid,Valid,0.023994,0.023994,2024-11-12 21:57:22.670000+00:00,0.85,正常,完全可信（权重1.0）
6,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:26.570000+00:00,2.996974,3.837063,CHA,12_47,CHA/3.0,3.0,...,True,True,Valid,Valid,0.089941,0.089941,2024-11-12 22:58:26.040000+00:00,0.79,正常,完全可信（权重1.0）
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 21:29:06.100000+00:00,1.499694,3.801039,CHA,9_40,CHA/1.5,1.5,...,True,True,Valid,Valid,0.011993,0.011993,2024-10-23 21:29:05.590000+00:00,0.85,正常,完全可信（权重1.0）
232,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.530000+00:00,-2.999504,3.717166,DCH,9_44,DCH/-3.0,3.0,...,True,True,Valid,Valid,0.011993,0.011993,2024-10-23 22:29:48.980000+00:00,0.84,正常,完全可信（权重1.0）
233,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0,3.0,...,True,True,Valid,Valid,0.006000,0.006000,2024-10-23 23:30:52.300000+00:00,0.81,正常,完全可信（权重1.0）
235,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5,1.5,...,True,True,Valid,Valid,0.065992,0.065992,2024-10-24 05:23:31.420000+00:00,0.88,正常,完全可信（权重1.0）


In [5]:
# =============================================================================
# 10. R0结果按flag输出 + 按SOC / pulse / current分类绘图
# =============================================================================
# 说明：
#   1. R0 本身已经在 build_time_diff_sequence(df) 中计算完成；
#   2. 这里基于 filtered_pulse 输出 R0 结果和 flag 统计；
#   3. R0 单位从 Ohm 转为 mOhm；
#   4. 绘图分类方式参考：
#      10% / 50% / 90% SOC 分成三个子图；
#      每条线按 SOC + CHA/DCH + 电流大小 分类。

# -----------------------------
# 1. 选择R0结果来源
# -----------------------------
if "filtered_pulse" in globals():
    r0_source = filtered_pulse.copy()
elif "pulse_sequence" in globals():
    r0_source = pulse_sequence.copy()
else:
    raise NameError("请先运行前面的cell，生成 pulse_sequence 或 filtered_pulse。")

required_columns = [
    "SOH",
    "SOC",
    "Time",
    "Current",
    "Voltage",
    "Zustand",
    "Zustand/Current",
    "Nominal_Current_A",
    "R0",
    "R1",
    "R2",
    "Temperature",
    "T_start",
    "T_end",
    "T_mean",
    "Delta_T",
    "CC_Valid_10s",
    "CC_Valid_20s",
    "CC_10s_Status",
    "CC_20s_Status",
    "CC_Max_Deviation_10s_pct",
    "CC_Max_Deviation_20s_pct",
    "R0_Target_Time",
    "Pause_to_Pulse_Time_Diff_s",
    "R0_Quality"
]

missing_columns = [col for col in required_columns if col not in r0_source.columns]
if missing_columns:
    raise KeyError(f"R0结果缺少必要列: {missing_columns}")

# -----------------------------
# 2. 整理R0结果
# -----------------------------
r0_result = r0_source.copy()

r0_result["SOH"] = pd.to_numeric(r0_result["SOH"], errors="coerce")
r0_result["Current"] = pd.to_numeric(r0_result["Current"], errors="coerce")
r0_result["Nominal_Current_A"] = pd.to_numeric(
    r0_result["Nominal_Current_A"], errors="coerce"
)
r0_result["R0"] = pd.to_numeric(r0_result["R0"], errors="coerce")
r0_result["R1"] = pd.to_numeric(r0_result["R1"], errors="coerce")
r0_result["R2"] = pd.to_numeric(r0_result["R2"], errors="coerce")

# 电阻原单位为 Ohm，这里转换为 mOhm
r0_result["R0_mOhm"] = r0_result["R0"] * 1000
r0_result["R1_mOhm"] = r0_result["R1"] * 1000
r0_result["R2_mOhm"] = r0_result["R2"] * 1000

r0_result["R0_Quality"] = (
    r0_result["R0_Quality"]
    .fillna("缺失")
    .astype(str)
    .str.strip()
)

r0_result["Current_abs_A"] = r0_result["Current"].abs().round(1)

r0_result["Current_Label"] = r0_result["Nominal_Current_A"].map(
    lambda x: f"{x:.1f}A" if pd.notna(x) else "UnknownA"
)

r0_result["SOC_pulse_current"] = (
    r0_result["SOC"].astype(str)
    + " "
    + r0_result["Zustand"].astype(str)
    + " "
    + r0_result["Current_Label"]
)

r0_result["R0_Is_Valid"] = (
    r0_result["R0_mOhm"].notna()
    & np.isfinite(r0_result["R0_mOhm"])
)

# 逐条记录的R0置信度。即使前面cell尚未重跑，这里也会重新生成。
def classify_r0_confidence(row):
    if not row["R0_Is_Valid"]:
        return R0_CONFIDENCE_INVALID

    flags = {
        flag.strip()
        for flag in str(row["R0_Quality"]).split("；")
        if flag.strip()
    }

    if flags and flags.issubset(TRUSTED_R0_QUALITY_FLAGS):
        return R0_CONFIDENCE_FULL

    return R0_CONFIDENCE_REVIEW

r0_result["R0_Confidence"] = r0_result.apply(
    classify_r0_confidence,
    axis=1
)

result_display_columns = [
    "SOH",
    "SOC",
    "Time",
    "Zustand",
    "Current",
    "Nominal_Current_A",
    "Voltage",
    "R0",
    "R0_mOhm",
    "R1",
    "R1_mOhm",
    "R2",
    "R2_mOhm",
    "Temperature",
    "T_start",
    "T_end",
    "T_mean",
    "Delta_T",
    "CC_Valid_10s",
    "CC_Valid_20s",
    "CC_10s_Status",
    "CC_20s_Status",
    "CC_Max_Deviation_10s_pct",
    "CC_Max_Deviation_20s_pct",
    "R0_Target_Time",
    "Pause_to_Pulse_Time_Diff_s",
    "R0_Quality",
    "R0_Confidence",
    "SOC_pulse_current",
    "File"
]

print("R0 / R1 / R2结果：按 R0_Quality / SOC / SOH 排序")
display(
    r0_result[result_display_columns]
    .sort_values(
        ["R0_Quality", "SOC", "SOH", "Zustand", "Current"],
        ascending=[True, True, False, True, True]
    )
    .reset_index(drop=True)
)

# -----------------------------
# 3. R0_Quality统计
# -----------------------------
# 若一条记录包含多个以“；”分隔的flag，会拆开后分别统计。
r0_quality_summary = (
    r0_result
    .assign(R0_Quality_Flag=r0_result["R0_Quality"].str.split("；"))
    .explode("R0_Quality_Flag")
)

r0_quality_summary["R0_Quality_Flag"] = (
    r0_quality_summary["R0_Quality_Flag"]
    .fillna("缺失")
    .astype(str)
    .str.strip()
)

r0_quality_summary = (
    r0_quality_summary
    .groupby("R0_Quality_Flag", dropna=False)
    .agg(
        Flag_Count=("R0_Quality_Flag", "size"),
        Valid_R0_Count=("R0_Is_Valid", "sum")
    )
    .reset_index()
    .sort_values("Flag_Count", ascending=False)
)

r0_quality_summary["Percent_of_segments"] = (
    r0_quality_summary["Flag_Count"] / len(r0_result) * 100
)

# flag级别的建议置信度：
# 3（正常）、1（后段不稳但早期窗口稳定）、4（首点0且电压跳变）
# 都作为干净标签，建议权重1.0。
r0_quality_summary["R0_Confidence"] = np.select(
    [
        (
            r0_quality_summary["R0_Quality_Flag"].isin(
                TRUSTED_R0_QUALITY_FLAGS
            )
            & r0_quality_summary["Valid_R0_Count"].gt(0)
        ),
        r0_quality_summary["Valid_R0_Count"].eq(0)
    ],
    [
        R0_CONFIDENCE_FULL,
        R0_CONFIDENCE_INVALID
    ],
    default=R0_CONFIDENCE_REVIEW
)

print("R0_Quality统计：")
display(r0_quality_summary)

# -----------------------------
# 4. 按图示方式绘图：SOH vs R0 / R1 / R2，分SOC子图，按 SOC / pulse / current 分线
# -----------------------------
# 绘制正常R0，以及“后段电流不稳定但早期拟合窗口稳定”的有效R0
r0_plot_data = r0_result[
    r0_result["R0_Is_Valid"]
    & r0_result["R0_Quality"].isin([
        "正常",
        R0_EARLY_WINDOW_FLAG
    ])
].copy()

if r0_plot_data.empty:
    print("没有可绘制的有效R0 / R1 / R2数据。")
else:
    soc_plot_order = ["10%", "50%", "90%"]
    available_soc_order = [
        soc for soc in soc_plot_order
        if soc in r0_plot_data["SOC"].astype(str).unique()
    ]

    # 如果出现了不在默认顺序中的SOC，也保留在后面
    extra_soc = [
        soc for soc in r0_plot_data["SOC"].astype(str).unique()
        if soc not in available_soc_order
    ]
    available_soc_order = available_soc_order + sorted(extra_soc)

    resistance_columns = {
        "R0": "R0_mOhm",
        "R1": "R1_mOhm",
        "R2": "R2_mOhm"
    }

    # R0、R1、R2统一使用同一个Y轴范围，便于直接比较大小。
    all_resistance_values = r0_plot_data[
        list(resistance_columns.values())
    ].to_numpy().ravel()
    all_resistance_values = all_resistance_values[
        np.isfinite(all_resistance_values)
    ]
    y_min = all_resistance_values.min()
    y_max = all_resistance_values.max()
    y_padding = 0.05 * (y_max - y_min) if y_max > y_min else 1.0
    common_y_range = [y_min - y_padding, y_max + y_padding]

    labels = sorted(r0_plot_data["SOC_pulse_current"].unique())
    colors = px.colors.qualitative.Plotly
    color_map = {
        label: colors[index % len(colors)]
        for index, label in enumerate(labels)
    }

    for resistance_name, resistance_column in resistance_columns.items():
        plot_data = r0_plot_data[
            np.isfinite(r0_plot_data[resistance_column])
        ].copy()

        if plot_data.empty:
            print(f"没有可绘制的有效{resistance_name}数据。")
            continue

        fig = make_subplots(
            rows=1,
            cols=len(available_soc_order),
            subplot_titles=[f"{soc} SOC" for soc in available_soc_order],
            shared_yaxes=True,
            horizontal_spacing=0.08
        )

        shown_legend = set()

        for col_idx, soc in enumerate(available_soc_order, start=1):
            soc_data = plot_data[
                plot_data["SOC"].astype(str) == soc
            ].copy()

            # 按SOH从高到低排序，配合反向x轴，视觉上和示例图一致
            soc_data = soc_data.sort_values(
                ["SOC_pulse_current", "SOH"],
                ascending=[True, False]
            )

            for label, group in soc_data.groupby("SOC_pulse_current", sort=True):
                group = group.sort_values("SOH", ascending=False)

                fig.add_trace(
                    go.Scatter(
                        x=group["SOH"],
                        y=group[resistance_column],
                        mode="lines+markers",
                        name=label,
                        legendgroup=label,
                        line=dict(color=color_map[label]),
                        showlegend=label not in shown_legend
                    ),
                    row=1,
                    col=col_idx
                )
                shown_legend.add(label)

            fig.update_xaxes(
                title_text="SOH (%)",
                autorange="reversed",
                row=1,
                col=col_idx
            )

            fig.update_yaxes(
                title_text=f"{resistance_name} (mOhm)",
                range=common_y_range,
                row=1,
                col=col_idx
            )

        fig.update_layout(
            title=f"SOH vs SOC 10%, 50%, 90% at {resistance_name}",
            legend_title_text="SOC / pulse / current",
            width=1500,
            height=600,
            template="plotly_white"
        )

        fig.show()


R0 / R1 / R2结果：按 R0_Quality / SOC / SOH 排序


,SOH,SOC,Time,Zustand,Current,Nominal_Current_A,Voltage,R0,R0_mOhm,R1,...,CC_10s_Status,CC_20s_Status,CC_Max_Deviation_10s_pct,CC_Max_Deviation_20s_pct,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence,SOC_pulse_current,File
0,81.7,50%,2025-06-26 20:02:05.060000+00:00,CHA,2.999851,3.0,3.879424,NaN,NaN,NaN,...,Not evaluated,Not evaluated,NaN,NaN,2025-06-26 20:01:53.770000+00:00,11.66,pause结束点到pulse首点时间差>5s,不可用（权重0.0）,50% CHA 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM34_...
1,81.7,90%,2025-06-26 12:20:22.870000+00:00,CHA,2.370311,3.0,4.199966,NaN,NaN,NaN,...,Not evaluated,Not evaluated,NaN,NaN,2025-06-26 12:20:04.430000+00:00,18.80,pause结束点到pulse首点时间差>5s,不可用（权重0.0）,90% CHA 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM34_...
2,96.2,10%,2024-09-06 06:45:06.580000+00:00,CHA,1.496906,1.5,3.322726,0.018759,18.759214,0.016940,...,Valid,Valid,0.144015,0.144015,2024-09-06 06:45:06.110000+00:00,0.82,正常,完全可信（权重1.0）,10% CHA 1.5A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...
3,96.2,10%,2024-09-06 08:46:53.490000+00:00,CHA,2.999132,3.0,3.368979,0.018930,18.929873,0.015899,...,Valid,Valid,0.017987,0.017987,2024-09-06 08:46:53+00:00,0.84,正常,完全可信（权重1.0）,10% CHA 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...
4,92.2,10%,2024-10-24 05:23:32.010000+00:00,CHA,1.498525,1.5,3.332511,0.019617,19.616687,0.017237,...,Valid,Valid,0.065992,0.065992,2024-10-24 05:23:31.420000+00:00,0.88,正常,完全可信（权重1.0）,10% CHA 1.5A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,77.8,50%,2025-12-14 23:26:48.130000+00:00,CHA,2.993016,3.0,3.878757,0.026168,26.167756,0.010142,...,Valid,Insufficient duration,0.161993,NaN,2025-12-14 23:26:47.680000+00:00,0.77,首点0且电压跳变,完全可信（权重1.0）,50% CHA 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM52_...
165,79.4,90%,2025-10-04 15:05:44.780000+00:00,CHA,1.497715,1.5,4.151712,0.041363,41.362918,0.019682,...,Valid,Insufficient duration,0.078021,NaN,2025-10-04 15:05:44.320000+00:00,0.76,首点0且电压跳变,完全可信（权重1.0）,90% CHA 1.5A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM44_...
166,78.6,90%,2025-11-09 20:12:28.080000+00:00,CHA,1.493757,1.5,4.145041,0.032455,32.455231,0.031511,...,Valid,Valid,0.210322,0.210322,2025-11-09 20:12:27.730000+00:00,0.75,首点0且电压跳变,完全可信（权重1.0）,90% CHA 1.5A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM48_...
167,78.6,90%,2025-11-09 21:13:10.940000+00:00,DCH,-2.991047,3.0,3.982345,0.032661,32.660834,0.024624,...,Valid,Insufficient duration,0.171227,NaN,2025-11-09 21:13:10.630000+00:00,0.74,首点0且电压跳变,完全可信（权重1.0）,90% DCH 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM48_...


R0_Quality统计：


,R0_Quality_Flag,Flag_Count,Valid_R0_Count,Percent_of_segments,R0_Confidence
1,正常,155,155,91.715976,完全可信（权重1.0）
2,首点0且电压跳变,12,12,7.100592,完全可信（权重1.0）
0,pause结束点到pulse首点时间差>5s,2,0,1.183432,不可用（权重0.0）
